# Lab 02 — Clustering & Anomaly Detection

## Business Context

Still at **ShopSmart**, the marketing team has two new requests:

1. **Segment customers** into meaningful groups to personalise campaigns *(Clustering)*
2. **Flag suspicious transactions** that might be fraud or data errors *(Anomaly Detection)*

This time we have **no labels** — we don't know in advance what the groups are or which transactions are anomalous. This is **unsupervised learning**.

---

## Learning Objectives

- Understand the difference between supervised and unsupervised learning
- Apply K-Means clustering and choose `k` with the Elbow Method
- Visualise and interpret customer segments
- Use Isolation Forest to detect anomalous transactions
- Communicate findings in a business-friendly way

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Ready!')

---

## Part 1 — Clustering: Customer Segmentation

### 1.1 The RFM Framework

A classic marketing approach is to describe customers using **RFM**:

| Metric | Meaning |
|---|---|
| **R**ecency | How recently did the customer buy? (lower = better) |
| **F**requency | How often do they buy? |
| **M**onetary | How much do they spend? |

In [ ]:
n = 400

# Simulate 4 distinct customer groups with different RFM profiles
segments_raw = [
    # (recency_mean, recency_std, freq_mean, freq_std, monetary_mean, monetary_std, n_customers, label)
    (10,  5,  20, 4,  500, 80,  80, 'Champions'),
    (45, 10,   8, 2,  150, 40, 120, 'At-Risk'),
    (90, 20,   3, 1,   60, 20, 100, 'Lost'),
    (20,  8,  12, 3,  280, 60, 100, 'Loyal'),
]

dfs = []
for r_m, r_s, f_m, f_s, m_m, m_s, n_seg, label in segments_raw:
    df_seg = pd.DataFrame({
        'recency_days': np.random.normal(r_m, r_s, n_seg).clip(1, 180).astype(int),
        'frequency':    np.random.normal(f_m, f_s, n_seg).clip(1, 40).astype(int),
        'monetary_eur': np.random.normal(m_m, m_s, n_seg).clip(10, 1000).round(2),
        'true_segment': label
    })
    dfs.append(df_seg)

df_rfm = pd.concat(dfs, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(df_rfm.shape)
df_rfm.head()

### 1.2 Scale the Features

> K-Means uses Euclidean distance — features must be on the same scale!

In [ ]:
features_rfm = ['recency_days', 'frequency', 'monetary_eur']
X_rfm = df_rfm[features_rfm].copy()

scaler = StandardScaler()
X_rfm_scaled = scaler.fit_transform(X_rfm)

print('Scaled shape:', X_rfm_scaled.shape)

### 1.3 Choose the Number of Clusters — Elbow Method

In [ ]:
inertias = []
silhouette_scores = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_rfm_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_rfm_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(k_range), inertias, 'bo-')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_title('Elbow Method')

axes[1].plot(list(k_range), silhouette_scores, 'go-')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Scores (higher is better)')

plt.tight_layout()
plt.show()

> **What do you notice?** Look for the 'elbow' — the point where adding more clusters gives diminishing returns.

### 1.4 Fit K-Means with k=4

In [ ]:
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df_rfm['cluster'] = kmeans.fit_predict(X_rfm_scaled)

print('Cluster distribution:')
print(df_rfm['cluster'].value_counts().sort_index())

### 1.5 Profile the Clusters

In [ ]:
profile = df_rfm.groupby('cluster')[features_rfm].mean().round(1)
print(profile)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
palette = sns.color_palette('tab10', k)

for ax, feat in zip(axes, features_rfm):
    means = profile[feat]
    ax.bar(means.index, means.values, color=palette)
    ax.set_xlabel('Cluster')
    ax.set_ylabel(feat)
    ax.set_title(f'Avg {feat} per Cluster')

plt.tight_layout()
plt.show()

### 1.6 Visualise with PCA (2D projection)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_rfm_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                      c=df_rfm['cluster'], cmap='tab10', alpha=0.6, s=30)
plt.legend(*scatter.legend_elements(), title='Cluster')
plt.title(f'K-Means Clusters (PCA 2D) — explained variance: {pca.explained_variance_ratio_.sum():.1%}')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

### 1.7 Assign Business Labels

Inspect the `profile` table above and assign a name to each cluster.

In [ ]:
# TODO: update this mapping based on the cluster profiles you see above
cluster_names = {
    0: 'Cluster 0 — ???',
    1: 'Cluster 1 — ???',
    2: 'Cluster 2 — ???',
    3: 'Cluster 3 — ???',
}

df_rfm['segment_name'] = df_rfm['cluster'].map(cluster_names)
print(df_rfm['segment_name'].value_counts())

---

## Part 2 — Anomaly Detection: Flagging Suspicious Transactions

### 2.1 Generate Transaction Data

Most transactions are normal. A small fraction are anomalous (unusually large amounts, odd timing, etc.).

In [ ]:
n_normal = 950
n_anomaly = 50

# Normal transactions
normal = pd.DataFrame({
    'amount_eur':    np.random.normal(75, 30, n_normal).clip(1, 250).round(2),
    'hour_of_day':  np.random.choice(range(8, 22), n_normal),   # business hours
    'items_count':  np.random.poisson(3, n_normal).clip(1, 15),
    'is_anomaly':    0
})

# Anomalous transactions
anomaly = pd.DataFrame({
    'amount_eur':   np.random.choice(
        np.concatenate([
            np.random.normal(600, 100, n_anomaly // 2).clip(400, 900),  # very high amounts
            np.random.uniform(0.01, 0.5, n_anomaly // 2)                # suspiciously small
        ])
    ).round(2),
    'hour_of_day':  np.random.choice([1, 2, 3, 4, 5], n_anomaly),      # middle of the night
    'items_count':  np.random.choice([0, 50, 100], n_anomaly),
    'is_anomaly':    1
})

df_txn = pd.concat([normal, anomaly], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Transactions: {len(df_txn)} | Anomalies: {df_txn["is_anomaly"].sum()}')
df_txn.head()

### 2.2 Visualise the Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['amount_eur', 'hour_of_day', 'items_count']):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        subset = df_txn[df_txn['is_anomaly'] == label]
        ax.hist(subset[col], bins=30, alpha=0.6, label='Normal' if label == 0 else 'Anomaly', color=color)
    ax.set_xlabel(col)
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

### 2.3 Train an Isolation Forest

**How it works:** Isolation Forest randomly splits the data. Anomalies are isolated in fewer splits (shorter path length) because they are rare and far from the rest.

In [ ]:
feature_cols = ['amount_eur', 'hour_of_day', 'items_count']
X_txn = df_txn[feature_cols]

iso_forest = IsolationForest(
    contamination=0.05,  # we expect ~5% anomalies
    random_state=42
)

# Returns 1 (normal) or -1 (anomaly)
raw_pred = iso_forest.fit_predict(X_txn)
df_txn['predicted_anomaly'] = (raw_pred == -1).astype(int)

print('Predicted anomalies:', df_txn['predicted_anomaly'].sum())

### 2.4 Evaluate the Results

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(classification_report(
    df_txn['is_anomaly'],
    df_txn['predicted_anomaly'],
    target_names=['Normal', 'Anomaly']
))

In [ ]:
cm = confusion_matrix(df_txn['is_anomaly'], df_txn['predicted_anomaly'])
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anomaly']).plot()
plt.title('Isolation Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

### 2.5 Inspect Flagged Transactions

In [ ]:
flagged = df_txn[df_txn['predicted_anomaly'] == 1].sort_values('amount_eur', ascending=False)
print(f'Flagged: {len(flagged)} transactions')
flagged.head(10)

---

## Your Turn! 🧪

**Exercise A — Clustering:**
Try `k=3` and `k=5`. Compare Silhouette Scores. Is 4 really the best number of segments for this data?

**Exercise B — Clustering:**
Change `contamination` in Isolation Forest from `0.05` to `0.10`. How does this affect the number of flagged transactions and the precision/recall?

**Exercise C — Business thinking:**
You have 4 customer segments. Write a short (3-sentence) marketing recommendation for each segment. What action would you take for 'Lost' customers vs 'Champions'?

In [ ]:
# Your code here


---

## Summary

| | Clustering | Anomaly Detection |
|---|---|---|
| **Goal** | Find natural groups | Find rare outliers |
| **Algorithm** | K-Means | Isolation Forest |
| **Labels needed?** | No | No |
| **Key challenge** | Choosing k | Setting contamination threshold |

**Next session:** We'll move to text data — using classic NLP to analyse customer reviews.